In [1]:
import sys; sys.path.append('..')
from osp import *
import html

txt = """
These words, it seems to me, give us a particular picture of the
essence of human language. It is this: the individual words in language
name objects—sentences are combinations of such names.——In this
picture of language we find the roots of the following idea: Every word
has a meaning. This meaning is correlated with the word. It is the
object for which the word stands.


Only Marxism can give us an account of the essential mystery of the cultural past, which, like Tiresias drinking the blood, is momentarily returned to life and warmth and once more allowed to speak, and to deliver its long-forgotten message in surroundings utterly alien to it.

There is a breathlessness about this shift from the normal object-oriented activity of the mind to such dialectical self-consciousness – something of the sickening shudder we feel in an elevator’s fall or in the sudden dip in an airliner.
""" * 7



In [10]:
doc = get_nlp_doc(txt)
sent = doc.sentences[-1]
tree = get_sent_tree(sent)

sum(len(sent.words) for sent in doc.sentences)

1239

In [11]:
# # extract_syntax_feats(doc)
# for id,docstr in STASH_SLICES_NLP.items():
#     if random.random() < 0.9:
#         doc = stanza.Document.from_serialized(docstr)
#         sent = doc.sentences[0]
#         break

In [13]:
extract_slice_feats(doc, normalize=True)

{'pos_DT': 146.89265536723164,
 'pos_NNS': 45.19774011299435,
 'pos_PRP': 50.84745762711865,
 'pos_VBZ': 45.19774011299435,
 'pos_IN': 135.59322033898306,
 'pos_VB': 22.598870056497177,
 'pos_JJ': 73.44632768361582,
 'pos_NN': 192.090395480226,
 'pos_VBP': 16.949152542372882,
 'pos_VBG': 11.299435028248588,
 'pos_VBN': 28.248587570621467,
 'pos_WDT': 11.299435028248588,
 'pos_RB': 22.598870056497177,
 'pos_NNP': 11.299435028248588,
 'pos_MD': 5.649717514124294,
 'pos_CC': 22.598870056497177,
 'pos_RBR': 5.649717514124294,
 'pos_TO': 11.299435028248588,
 'pos_PRP$': 5.649717514124294,
 'pos_HYPH': 16.949152542372882,
 'pos_EX': 5.649717514124294,
 'pos_POS': 5.649717514124294,
 'deprel_det': 141.24293785310735,
 'deprel_nsubj': 73.44632768361582,
 'deprel_punct': 124.29378531073446,
 'deprel_parataxis': 16.949152542372882,
 'deprel_case': 135.59322033898306,
 'deprel_obl': 45.19774011299435,
 'deprel_root': 39.54802259887006,
 'deprel_iobj': 11.299435028248588,
 'deprel_amod': 84.745762

In [14]:
# extract_slice_feats(docstr)

In [6]:
def get_node_path_to_root(node):
    path = [node.label()]
    while node.parent() is not None:
        path.append(node.parent().label())
        node = node.parent()
    return path

In [ ]:
def is_preterminal(node):
    return node.height() == 2

def get_phrase_counts(tree):
    counter = Counter()
    for x in tree.subtrees():
        if is_preterminal(x):
            terminal = x.leaves()[0]
            if not terminal or not terminal[0].isalpha():
                continue
            for y in get_node_path_to_root(x):
                if y not in POS2DESC:
                    counter[y] += 1
    return counter

get_phrase_counts(tree)

Counter({'NP': 130,
         'PP': 92,
         'S': 54,
         'VP': 53,
         'ROOT': 40,
         'SBAR': 14,
         'ADJP': 2})

In [8]:
counter

NameError: name 'counter' is not defined

In [ ]:
# Get preterminal nodes (nodes whose children are all leaves/terminals)
preterminals = [node for node in tree.subtrees() if node.height() == 2]
for pt in preterminals:
    print(pt.label(), pt.leaves())


PRP ['It']
VBZ ['is']
DT ['the']
NN ['object']
IN ['for']
WDT ['which']
DT ['the']
NN ['word']
VBZ ['stands']
. ['.']


In [ ]:
'$'.isalpha()

False

In [ ]:
stopxx

NameError: name 'stopxx' is not defined

In [ ]:
def get_syntax_df(sent):
    dfx=get_clauses_v2(sent).sort_values('word_i')
    dfx['clause_head'] = dfx['clause_head_id']-1
    dfx = dfx.drop(columns=['clause_head_id'])
    dfx['word_head_dist'] = (dfx['word_head'] - dfx['word_i']).abs()
    return dfx


In [ ]:
df=get_syntax_df(sent)

In [ ]:
df.mean(numeric_only=True)

clause_i          1.45
clause_id         1.75
word_i            9.50
word_head         9.80
word_depth        2.05
clause_depth      0.25
clause_head       7.00
word_head_dist    3.10
dtype: float64

In [ ]:
df

,clause_i,clause_id,clause_type,clause_deprel,word_i,word,word_pos,word_deprel,word_head,word_depth,clause_depth,clause_head,word_head_dist
0,0,2,main,root,0,These,DT,det,2,2,0,8,2
1,0,2,main,root,1,words,NNS,nsubj,9,1,0,8,8
2,0,2,main,root,2,",",",",punct,2,2,0,8,0
3,1,1,sub,parataxis,3,it,PRP,nsubj,5,2,1,4,2
4,1,1,sub,parataxis,4,seems,VBZ,parataxis,9,1,1,4,5
5,1,1,sub,parataxis,5,to,IN,case,7,3,1,4,2
6,1,1,sub,parataxis,6,me,PRP,obl,5,2,1,4,1
7,1,1,sub,parataxis,7,",",",",punct,5,2,1,4,2
8,2,2,main,root,8,give,VB,root,0,0,0,8,8
9,2,2,main,root,9,us,PRP,iobj,9,1,0,8,0


In [ ]:
get_clause_form(sent)

'(IC(DC))'

In [ ]:
def extract_syntax_feats_sent(sent, incl_formula=True, max_n_clauses=3):
    df = get_syntax_df(sent)
    df_clause = df.drop_duplicates('clause_id')
    clause_type_counts = df_clause.groupby('clause_type').size()
    num_ic = int(clause_type_counts.get('main',0))
    num_dc = int(clause_type_counts.get('sub',0))

    
    out_d = {}
    out_d['IC']=num_ic
    out_d['DC']=num_dc
    out_d['C']=df.clause_id.nunique()
    out_d['C*']=df.clause_i.nunique()

    avg_s = df.max(numeric_only=True)
    out_d['Wd'] = int(avg_s['word_depth'])
    out_d['Cd'] = int(avg_s['clause_depth'])

    if out_d['C'] < max_n_clauses:
        out_d[get_clause_form(sent)]=1
    
    # df_words = df[df.word_deprel!='punct']
    # out_d['num_words_DC']=len(df_words[df_words.clause_type=='sub'])
    # out_d['num_words_IC']=len(df_words[df_words.clause_type=='main'])
    return out_d

def extract_syntax_feats(doc):
    df = pd.DataFrame([extract_syntax_feats_sent(sent) for sent in doc.sentences])
    return {k:int(v) for k,v in df.sum().items()}

In [ ]:
extract_syntax_feats(doc)

{'IC': 5,
 'DC': 4,
 'C': 9,
 'C*': 13,
 'Wd': 17,
 'Cd': 4,
 '(IC(DC))': 4,
 '(IC)': 1}